# Example 2.3: 参数化工况 + 共享 base INP + 参数化材料模板

主 INP 只是一个工况文件, 它 `*INCLUDE` 了两样东西 —— 一个静态的 base INP,
和一个**同样需要被修改**的材料模板:

```
planar_stress_nested_scenario.inp     # 根, 含 {{load_magnitude}}
├── planar_stress_main.inp            # 静态, 125 KB, 全批次共用
└── material_template.inp             # 含 {{youngs_modulus}}
```

准备阶段对 include 树里的每个文件只判一次:

| 分类 | 判据 | 落盘 | 改写后的指令 |
| --- | --- | --- | --- |
| **待修改** | 自己(或它的后代)含 `{{}}` | 代入参数, 在 job 目录里生成**独立副本** | 裸文件名 |
| **静态共享** | 其余 | 原地不动, 不复制 | 绝对路径 |

所以这个例子里, 每个 job 目录会有两个 INP: job 自己的主 INP, 和一份属于它自己的
`material_template.inp`; 而 125 KB 的 base INP 四个 job 共指同一份。

这里根 INP 和材料模板都需要修改 —— 这没有问题, 根 INP 本来就是每 job 一份的。
实践中当然可以只保留一个待修改的模板, 分类规则不变。

用法仍然和 `01_Batch_wo_include.ipynb` 一样: `kind="inp_based"` + `generate_from_array`,
`param_names` 覆盖整棵树里出现的所有 `{{}}` 即可(缺哪个会在准备阶段直接报错, 不会生成半成品)。

In [1]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec
from ABQflow import generate_from_array, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()

In [2]:
# youngs_modulus 来自 material_template.inp, load_magnitude 来自根 INP —— 一个参数字典管整棵树
param_names = ['youngs_modulus', 'load_magnitude']
param_values = np.array([
	[200000, 2000],
	[210000, 3000],
	[220000, 4000],
	[230000, 5000]
])

base_job_spec = JobSpec(
	job_name = "planar_stress_nested",
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "inp_based",
		# 工况文件: 自己含 {{load_magnitude}}, 并 *INCLUDE 了
		# planar_stress_main.inp(静态) 和 material_template.inp(待修改)
		source_path = "./examples/cae_file/planar_stress_nested_scenario.inp",
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	],
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

spec_list = generate_from_array(
	samples_array = param_values,
	param_names = param_names,
	base_spec  = base_job_spec
)

import pprint

pprint.pprint(spec_list)

[JobSpec(job_name='planar_stress_nested_0001',
         workflow='modular',
         preparation=PreparationSpec(kind='inp_based',
                                     source_path='./examples/cae_file/planar_stress_nested_scenario.inp',
                                     params={'load_magnitude': 2000.0,
                                             'youngs_modulus': 200000.0},
                                     options={}),
         preflight=None,
         monolithic_script=None,
         monolithic_params={},
         pre_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_total_mass.py',
                                  tasks=[{'result_name': 'total_mass'}])],
         post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                   tasks=[{'result_name': 'max_stress_mises'},
                                          {'result_name': 'max_displacement'}])],
         subroutine=None,
         meta={}

In [3]:
processor = BatchAbaqusProcessor(
	batch_data = spec_list,
	base_output_dir = os.path.join(CWD, "examples/02_BatchParameterizedJob/output_nested"),
	cpus_per_job = 12,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes = processor.run_batch(
	num_parallel_jobs = 2,
)

Output()

In [5]:
outcomes

[JobOutcome(job_name='planar_stress_nested_0001', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 4525.26025390625, 'max_displacement': 4.189039707183838}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/02_BatchParameterizedJob/output_nested\\planar_stress_nested_0001', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1788594191.2668426, 'ended_at': 1788594191.2698429, 'duration_s': 0.0030002593994140625, 'error': None}, {'phase': 'pre_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788594191.2698429, 'ended_at': 1788594193.8734372, 'duration_s': 2.6035943031311035, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1788594193.8734372, 'ended_at': 1788594243.0503461, 'duration_s': 49.17690896987915, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788594243.0503461, 'ended_at': 178859

In [6]:
Y = degenerate_from_array(
	outcomes = outcomes,
	output_names = ["total_mass", "max_stress_mises", "max_displacement"],
)
print(f"output: {Y}")

output: [[3.20662623e-04 4.52526025e+03 4.18903971e+00]
 [3.20662623e-04 6.78789062e+03 5.98434210e+00]
 [3.20662623e-04 9.05052051e+03 7.61643553e+00]
 [3.20662623e-04 1.13131514e+04 9.10660839e+00]]


In [7]:
# 验证: 待修改的模板变成了每 job 一份的副本, 静态 base INP 仍然共享
for oc in outcomes[:2]:
	job_inp = os.path.join(oc.output_dir, oc.job_name + ".inp")
	print("=" * 70)
	print(oc.job_name)

	print("  job 目录里的 INP 文件:")
	for name in sorted(os.listdir(oc.output_dir)):
		if name.endswith(".inp"):
			size = os.path.getsize(os.path.join(oc.output_dir, name))
			print(f"    {name:<45s} {size:>9,d} bytes")

	with open(job_inp, encoding="utf-8", errors="replace") as f:
		text = f.read()
	print("  主 INP 里的 *INCLUDE 指令:")
	for line in text.splitlines():
		if line.lstrip().lower().startswith("*include"):
			print("    " + line)

	# 这份是每个 job 自己的副本, 里面的 {{youngs_modulus}} 已经代入
	local = os.path.join(oc.output_dir, "material_template.inp")
	print("  它自己那份 material_template.inp 的材料段:")
	with open(local, encoding="utf-8", errors="replace") as f:
		for line in f:
			if not line.startswith("**"):
				print("    " + line.rstrip())

shared = "./examples/cae_file/planar_stress_main.inp"
print("=" * 70)
print(f"共享的 base INP: {os.path.getsize(shared):,d} bytes —— 四个 job 共指这一份,")
print("既没有被复制进任何 job 目录, 也没有被内联进主 INP。")


planar_stress_nested_0001
  job 目录里的 INP 文件:
    material_template.inp                               253 bytes
    planar_stress_nested_0001.inp                       882 bytes
  主 INP 里的 *INCLUDE 指令:
    *Include, input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples\cae_file\planar_stress_main.inp
    *Include, input=material_template.inp
  它自己那份 material_template.inp 的材料段:
    *Material, name=Material
    *Density
     2.7e-09,
    *Elastic
    200000.0, 0.3
planar_stress_nested_0002
  job 目录里的 INP 文件:
    material_template.inp                               253 bytes
    planar_stress_nested_0002.inp                       882 bytes
  主 INP 里的 *INCLUDE 指令:
    *Include, input=c:\SJTU\Projects_Code\24_Abaqus_Pack\examples\cae_file\planar_stress_main.inp
    *Include, input=material_template.inp
  它自己那份 material_template.inp 的材料段:
    *Material, name=Material
    *Density
     2.7e-09,
    *Elastic
    210000.0, 0.3
共享的 base INP: 125,469 bytes —— 四个 job 共指这一份,
既没有被复制进任何 job 目录, 也没有被内联进主 